# Mini-Project 1 — Data Exploration (NYC Taxi)
### Databricks + Snowflake 70-Hour Programme | ExcelR × KIIT

**Name:** Ayush Raj  
**Roll number:** 2305122  
**Date started:** 16/08/2026

---
**Before you start:**
1. Set `SEED` in Cell 2 to the **last 4 digits of your roll number**. Do not change it later.
2. Run every cell top to bottom. Never skip Cell 2 — it builds *your* copy of the data.
3. Fill in every `# TODO`. Delete nothing.
4. Write your answer as a short comment under each result. A number with no sentence earns half marks.

## Cell 1 — Imports and sanity check
You'll know it worked when the schema prints 6 columns and the count is a five-digit number.

In [0]:
from pyspark.sql import functions as F

base = spark.table("samples.nyctaxi.trips")
base.printSchema()
print("Rows in the shared source table:", base.count())
display(base.limit(5))

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)

Rows in the shared source table: 21932


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53.000Z,2016-02-13T21:57:15.000Z,1.4,8.0,10103,10110
2016-02-13T18:29:09.000Z,2016-02-13T18:37:23.000Z,1.31,7.5,10023,10023
2016-02-06T19:40:58.000Z,2016-02-06T19:52:32.000Z,1.8,9.5,10001,10018
2016-02-12T19:06:43.000Z,2016-02-12T19:20:54.000Z,2.3,11.5,10044,10111
2016-02-23T10:27:56.000Z,2016-02-23T10:58:33.000Z,2.6,18.5,10199,10022


## Cell 2 — Build YOUR dataset  ⚠️ EDIT THE SEED, CHANGE NOTHING ELSE

This cell deterministically produces a variant of the source data that belongs to you alone.
Same seed ⇒ same rows, every single time. Different seed ⇒ different answers.

**You'll know it worked when:** the printed row count is close to, but not equal to, the count
from Cell 1, and it ends with your own digits — nobody else in the room will print the same number.

In [0]:
# ============================================================
# EDIT THIS LINE ONLY
SEED = 5122          # <-- last 4 digits of your roll number, e.g. SEED = 1742
# ============================================================

assert SEED != 0000, "Set SEED to the last 4 digits of your roll number before running."

_keyed = base.withColumn(
    "row_key",
    F.xxhash64(
        F.col("tpep_pickup_datetime").cast("string"),
        F.col("tpep_dropoff_datetime").cast("string"),
        F.col("trip_distance").cast("string"),
        F.col("fare_amount").cast("string"),
        F.col("pickup_zip").cast("string"),
        F.col("dropoff_zip").cast("string"),
        F.lit(SEED).cast("string"),
    ),
)

# 1) keep a seeded subset of the rows
_sampled = _keyed.filter(F.pmod(F.col("row_key"), F.lit(100)) >= 12)

# 2) blank out the fare on a small seeded set of rows
_nulled = _sampled.withColumn(
    "fare_amount",
    F.when(F.pmod(F.col("row_key"), F.lit(97)) == 0, F.lit(None).cast("double"))
     .otherwise(F.col("fare_amount")),
)

# 3) re-insert a small seeded set of rows a second time
_dupes = _nulled.filter(F.pmod(F.col("row_key"), F.lit(67)) == 0)

trips = _nulled.unionByName(_dupes).drop("row_key")
trips.createOrReplaceTempView("my_trips")

print("SEED =", SEED)
print("Rows in MY dataset:", trips.count())

SEED = 5122
Rows in MY dataset: 19566


# PART A — Profile your data
*(unlocks after Day 6 — DataFrame API)*

Goal: describe what you have been given before you touch it. Do not clean anything yet.

In [0]:
# A1. How many rows and how many columns are in YOUR dataset?
print("Rows:", trips.count())
print("Columns:", len(trips.columns))

# Answer: 19,566 rows and 6 columns.

Rows: 19566
Columns: 6


In [0]:
# A2. Print the schema. In a comment, write the data type of every column in plain English,
#     e.g. "fare_amount is a double — money in US dollars".
trips.printSchema()

# Answer:
# tpep_pickup_datetime is a timestamp showing when the taxi trip started.
# tpep_dropoff_datetime is a timestamp showing when the taxi trip ended.
# trip_distance is a double representing the trip distance in miles.
# fare_amount is a double representing the taxi fare in US dollars.
# pickup_zip is an integer representing the pickup ZIP code.
# dropoff_zip is an integer representing the dropoff ZIP code.

root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)



In [0]:
# A3. What is the earliest and the latest pickup timestamp in your data?
#     Hint: F.min(...) and F.max(...) inside .agg()
display(
    trips.agg(
        F.min("tpep_pickup_datetime").alias("earliest_pickup"),
        F.max("tpep_pickup_datetime").alias("latest_pickup")
    )
)
# Answer: the data covers 2016-01-01T00:04:30 to 2016-02-29T23:51:20.

earliest_pickup,latest_pickup
2016-01-01T00:04:30.000Z,2016-02-29T23:51:20.000Z


In [0]:
# A4. How many rows are EXACT duplicates (every column identical to another row)?
#     Hint: compare .count() with .dropDuplicates().count()
total_rows = trips.count()
unique_rows = trips.dropDuplicates().count()
duplicate_rows = total_rows - unique_rows

print("Exact duplicate rows:", duplicate_rows)

# Answer: 291 duplicate rows


Exact duplicate rows: 291


In [0]:
# A5. How many NULLs are there in each column?
#     Hint: build one .agg() with F.count(F.when(F.col(c).isNull(), c)).alias(c) for each column,
#     or loop over trips.columns.
display(
    trips.agg(*[
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in trips.columns
    ])
)

# Answer: tpep_pickup_datetime = 0, tpep_dropoff_datetime = 0, trip_distance = 0, fare_amount = 196, pickup_zip = 0, dropoff_zip = 0.

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
0,0,0,196,0,0


In [0]:
# A6. Count the rows that look wrong even though they are not null:
#     (a) fare_amount <= 0
#     (b) trip_distance <= 0
#     (c) dropoff timestamp is earlier than or equal to the pickup timestamp
a = trips.filter(F.col("fare_amount") <= 0).count()

b = trips.filter(F.col("trip_distance") <= 0).count()

c = trips.filter(
    F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime")
).count()

print("(a) fare_amount <= 0:", a)
print("(b) trip_distance <= 0:", b)
print("(c) dropoff <= pickup:", c)

# Answer: (a) 10  (b) 68  (c) 1

(a) fare_amount <= 0: 10
(b) trip_distance <= 0: 68
(c) dropoff <= pickup: 1


# PART B — Clean your data
*(unlocks after Day 7 — transformations)*

Apply the five rules **in this order** and record how many rows survive each step.
This is your data-quality funnel and it is worth marks on its own.

| Step | Rule | Rows after |
|---|---|---|
| 0 | raw (`trips`) | |
| 1 | drop exact duplicates | |
| 2 | drop rows where `fare_amount` is NULL | |
| 3 | drop rows where `fare_amount <= 0` | |
| 4 | drop rows where `trip_distance <= 0` | |
| 5 | drop rows where dropoff <= pickup | |

Call the final result `clean` and register it as a temp view called `my_clean`.

In [0]:
step0 = trips
print("Step 0 - raw:", step0.count())

step1 = step0.dropDuplicates()
print("Step 1 - drop exact duplicates:", step1.count())

step2 = step1.filter(F.col("fare_amount").isNotNull())
print("Step 2 - drop NULL fare_amount:", step2.count())

step3 = step2.filter(F.col("fare_amount") > 0)
print("Step 3 - drop fare_amount <= 0:", step3.count())

step4 = step3.filter(F.col("trip_distance") > 0)
print("Step 4 - drop trip_distance <= 0:", step4.count())

step5 = step4.filter(
    F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")
)
print("Step 5 - drop dropoff <= pickup:", step5.count())

clean = step5
clean.createOrReplaceTempView("my_clean") 


Step 0 - raw: 19566
Step 1 - drop exact duplicates: 19275
Step 2 - drop NULL fare_amount: 19079
Step 3 - drop fare_amount <= 0: 19069
Step 4 - drop trip_distance <= 0: 19002
Step 5 - drop dropoff <= pickup: 19002


**B7 (written, 3–4 sentences).** For each of the five rules, say in one line *why* a real analyst
would drop those rows — and name one rule you think is arguable, and what you would do instead.
Write your answer in the cell below.

*Your answer here:*
Exact duplicates are dropped because duplicate rows can count the same trip more than once. NULL fare_amount rows are dropped because fare is needed for fare-based analysis, while fare_amount <= 0 rows are removed because they are invalid for normal taxi fares. trip_distance <= 0 rows are dropped because a valid trip should have a positive distance, and dropoff <= pickup rows are removed because they have an invalid trip duration. The NULL fare rule is arguable; instead, I would first check whether the missing fare can be recovered from another reliable source before dropping the row.

# PART C — Business questions
*(unlocks after Day 7)*

Use `clean` (or the `my_clean` view) for everything below. You may answer in PySpark or in `%sql` —
use at least one of each somewhere in this section.

In [0]:
# C1. Headline numbers: total trips, total fare collected, average fare, average trip distance.
#     Round money to 2 decimals and distance to 3.
result = clean.agg(
    F.count("*").alias("total_trips"),
    F.sum("fare_amount").alias("total_fare"),
    F.avg("fare_amount").alias("average_fare"),
    F.avg("trip_distance").alias("average_distance")
)

display(
    result.select(
        "total_trips",
        F.round("total_fare", 2).alias("total_fare"),
        F.round("average_fare", 2).alias("average_fare"),
        F.round("average_distance", 3).alias("average_distance")
    )
)
# Answer: There were 19,002 total trips, with $233,671.02 total fare collected. The average fare was $12.30 and the average trip distance was 2.852 miles.

total_trips,total_fare,average_fare,average_distance
19002,233671.02,12.3,2.852


In [0]:
# C2. Which HOUR OF THE DAY has the most pickups? Show all 24 hours ordered by trip count.
#     Hint: F.hour("tpep_pickup_datetime")
hourly = (
    clean
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
)

display(hourly)
# Answer: busiest hour is 19 with 1,250 trips.

pickup_hour,count
0,650
1,491
2,398
3,267
4,211
5,183
6,400
7,701
8,899
9,900


In [0]:
# C3. Top 5 pickup_zip values by number of trips. For each, also show average fare and average distance.
result = (
    clean
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("average_fare"),
        F.avg("trip_distance").alias("average_distance")
    )
    .orderBy(F.desc("trip_count"))
    .limit(5)
)

display(
    result.select(
        "pickup_zip",
        "trip_count",
        F.round("average_fare", 2).alias("average_fare"),
        F.round("average_distance", 3).alias("average_distance")
    )
)
# Answer: The top 5 pickup ZIP codes by trip count are 10001 (1,060 trips), 10003 (1,023), 10011 (991), 10021 (893), and 10018 (878).

pickup_zip,trip_count,average_fare,average_distance
10001,1060,10.67,2.223
10003,1023,10.95,2.303
10011,991,10.76,2.232
10021,893,10.21,2.042
10018,878,11.54,2.612


In [0]:
# C4. Create a column `fare_per_mile` = fare_amount / trip_distance.
#     Which 10 pickup zips have the HIGHEST average fare per mile,
#     counting only zips with at least 50 trips?
#     Hint: .groupBy(...).agg(...) then .filter(F.col("trips") >= 50)
result = (
    clean
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance"))
    .groupBy("pickup_zip")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_per_mile").alias("avg_fare_per_mile")
    )
    .filter(F.col("trip_count") >= 50)
    .orderBy(F.desc("avg_fare_per_mile"))
    .limit(10)
)

display(
    result.select(
        "pickup_zip",
        "trip_count",
        F.round("avg_fare_per_mile", 2).alias("avg_fare_per_mile")
    )
)
# Answer: The top 10 pickup ZIP codes by average fare per mile are 10003 (7.86), 10020 (7.56), 10017 (6.99), 10171 (6.62), 10111 (6.59), 10110 (6.46), 10167 (6.43), 10153 (6.34), 10103 (6.32), and 10021 (6.28).

pickup_zip,trip_count,avg_fare_per_mile
10003,1023,7.86
10020,404,7.56
10017,606,6.99
10171,175,6.62
10111,144,6.59
10110,661,6.46
10167,245,6.43
10153,359,6.34
10103,505,6.32
10021,893,6.28


In [0]:
# C5. Create a column `duration_min` = (dropoff - pickup) in minutes.
#     Hint: (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
#     (a) What is the average trip duration?
#     (b) Which DAY OF THE WEEK has the most trips? Hint: F.date_format(col, "EEEE")
duration = clean.withColumn(
    "duration_min",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

avg_duration = duration.agg(
    F.avg("duration_min").alias("avg_duration_min")
)

day_counts = (
    duration
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week")
    .count()
    .orderBy(F.desc("count"))
)

display(avg_duration)
display(day_counts)
# Answer: (a) 15.209 minutes  (b) Friday

avg_duration_min
15.208832403606648


day_of_week,count
Friday,3088
Saturday,2962
Thursday,2781
Sunday,2635
Wednesday,2566
Monday,2559
Tuesday,2411


In [0]:
# C6. Show the single longest trip by distance, and the single most expensive trip by fare.
#     Print the full row for each.
longest = clean.orderBy(F.desc("trip_distance")).limit(1)
most_expensive = clean.orderBy(F.desc("fare_amount")).limit(1)

print("Longest trip:")
display(longest)

print("Most expensive trip:")
display(most_expensive)
# Answer: The longest trip was 30.6 miles with a fare of $95, from pickup ZIP 11371 to dropoff ZIP 7114. The most expensive trip had a fare of $275, with a distance of 20.85 miles, from pickup ZIP 10013 to dropoff ZIP 7008.

Longest trip:


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-22T21:17:27.000Z,2016-02-22T22:00:58.000Z,30.6,95.0,11371,7114


Most expensive trip:


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-12T20:55:19.000Z,2016-02-12T21:52:38.000Z,20.85,275.0,10013,7008


# PART D — Your own question

In [0]:
# My question:
# Which day of the week has the highest average fare?

day_fares = (
    clean
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("average_fare")
    )
    .orderBy(F.desc("average_fare"))
)

display(
    day_fares.select(
        "day_of_week",
        "trip_count",
        F.round("average_fare", 2).alias("average_fare")
    )
)

# What I found:
# Thursday had the highest average fare at $12.66 across 2,781 trips.
# A taxi company would care because identifying days with higher average fares
# can help with revenue planning and driver allocation.

day_of_week,trip_count,average_fare
Thursday,2781,12.66
Friday,3088,12.57
Wednesday,2566,12.41
Tuesday,2411,12.32
Monday,2559,12.24
Sunday,2635,12.17
Saturday,2962,11.72


# PART E — Submission signature

Run the cell below **exactly as written** after `clean` exists. Copy the printed block into your
findings summary. It is the proof that these numbers came from your own dataset.

In [0]:
sig = clean.select(
    F.pmod(
        F.xxhash64(
            F.col("tpep_pickup_datetime").cast("string"),
            F.col("tpep_dropoff_datetime").cast("string"),
            F.col("trip_distance").cast("string"),
            F.col("fare_amount").cast("string"),
            F.col("pickup_zip").cast("string"),
            F.col("dropoff_zip").cast("string"),
        ),
        F.lit(1000003),
    ).alias("h")
).agg(
    F.count("*").alias("clean_rows"),
    F.sum("h").alias("dataset_signature"),
).collect()[0]

totals = clean.agg(
    F.round(F.sum("fare_amount"), 2).alias("total_fare"),
    F.round(F.avg("trip_distance"), 4).alias("avg_distance"),
).collect()[0]

print("=========== MINI-PROJECT 1 SIGNATURE ===========")
print("SEED              :", SEED)
print("CLEAN ROWS        :", sig["clean_rows"])
print("DATASET SIGNATURE :", sig["dataset_signature"])
print("TOTAL FARE        :", totals["total_fare"])
print("AVG DISTANCE      :", totals["avg_distance"])
print("================================================")

=========== MINI-PROJECT 1 SIGNATURE ===========
SEED              : 5122
CLEAN ROWS        : 19002
DATASET SIGNATURE : 9488604923
TOTAL FARE        : 233671.02
AVG DISTANCE      : 2.8524


## Finally
1. Open **Query History** (left sidebar), find one of your `groupBy` queries, open its **Query Profile**
   and take a screenshot showing rows read and time taken. Submit it with your notebook.
2. Export this notebook: **File → Export → HTML** (or `.ipynb`) and submit the file.
3. Submit your 1-page findings summary with the signature block pasted at the bottom.